In [1]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
MODEL_TYPE = "DeepCNNLSTM"  

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs\\ablations_hpo"
FILE_NAME = f"ablation_A2_1s3w_hpo_v1"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: ablation_A2_1s3w_hpo_v1
From path: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\ablations_hpo\ablation_A2_1s3w_hpo_v1.log


In [2]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'ablation_A2_1s3w_hpo_v1'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 200 trials.
Best value (Accuracy): 0.7782
Best Hyperparameters:
  outer_lr: 0.00034779613260399434
  wd: 7.612219508371405e-05
  ft_lr: 0.00560010547178871
  label_smooth: 0.38762498225796704


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [8]:
# FULL

#params_to_plot_ARCH = [
#    "cnn_base_filters", "lstm_hidden", "groupnorm_num_groups", "use_GlobalAvgPooling"
#]

params_to_plot_OPT = [
    "outer_lr", "wd", "label_smooth", "ft_lr"
]

#params_to_plot_MAML = [
#    "meta_batchsize", "maml_inner_steps", "maml_inner_steps_eval", "maml_alpha_init", "maml_alpha_init_eval","maml_use_lslr","use_lslr_at_eval","use_maml_msl"
#]

#params_to_plot_MOE_CORE = [
#    "num_experts", "MOE_top_k", "MOE_aux_coeff", "MOE_gate_temperature"
#]

#params_to_plot_MOE_AUX = [
#    "MOE_ctx_out_dim", "MOE_ctx_hidden_dim", "MOE_dropout"
#]

In [9]:
from optuna.visualization import plot_slice


In [10]:
fig_slice = plot_slice(study, params=params_to_plot_OPT)
fig_slice.show()

In [11]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
193,193,0.778229,0.2825,2026-04-19 23:28:51.846626,0 days 00:00:54.438148
185,185,0.772917,0.2225,2026-04-19 23:27:20.880551,0 days 00:00:56.361748
109,109,0.772500,0.2450,2026-04-19 23:15:11.872132,0 days 00:00:54.471833
145,145,0.772292,0.2350,2026-04-19 23:21:16.144891,0 days 00:00:56.577844
132,132,0.771771,0.2300,2026-04-19 23:19:44.417862,0 days 00:00:55.608643
161,161,0.771771,0.2275,2026-04-19 23:24:18.193048,0 days 00:00:56.610223
195,195,0.770313,0.2750,2026-04-19 23:28:52.139523,0 days 00:00:52.819324
155,155,0.770104,0.2600,2026-04-19 23:22:47.374445,0 days 00:00:54.315813
170,170,0.768750,0.2325,2026-04-19 23:25:49.331798,0 days 00:00:56.631978
167,167,0.768542,0.2175,2026-04-19 23:24:19.428691,0 days 00:00:55.114231


In [12]:
from optuna.trial import TrialState

# 1. Filter for only successfully completed trials
completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE]

# 2. Sort the filtered list (using reverse=True if you want the highest values)
top_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:10]

# 3. Print the params
for t in top_trials:
    print(t.params)

{'outer_lr': 0.00034779613260399434, 'wd': 7.612219508371405e-05, 'ft_lr': 0.00560010547178871, 'label_smooth': 0.38762498225796704}
{'outer_lr': 0.0001070872776586305, 'wd': 4.589625449793772e-05, 'ft_lr': 0.0015463018700855062, 'label_smooth': 0.16768517276400696}
{'outer_lr': 0.00023911982934860987, 'wd': 1.953354617385229e-05, 'ft_lr': 0.0013797361628352998, 'label_smooth': 0.056490898644469795}
{'outer_lr': 0.00011370016943229966, 'wd': 4.984659509657444e-05, 'ft_lr': 0.002077148709752472, 'label_smooth': 0.18723509486846068}
{'outer_lr': 0.00010785221463149262, 'wd': 4.7246888918068603e-05, 'ft_lr': 0.002737056019822687, 'label_smooth': 0.1951202787811779}
{'outer_lr': 0.00012108159197531468, 'wd': 4.647064348559639e-05, 'ft_lr': 0.002914279683011286, 'label_smooth': 0.15867890933341489}
{'outer_lr': 0.000308821801726551, 'wd': 3.046843860844335e-05, 'ft_lr': 0.0025865582472135345, 'label_smooth': 0.09844946450649186}
{'outer_lr': 0.0003112491418773761, 'wd': 2.3771691025432164e-